# Helpers for Testing

### Export archive as JSON

In [ ]:
# Takes an archive and saves it to a JSON file. The archive is a dictionary of solutions
import json

archive_to_save = extended
output_path = 'my_archive.json'

data = [
    {
        'sol': [int(s) for s in entry['sol']],
        'obj': [float(x) for x in entry['obj']],
    }
    for entry in archive_to_save.values()
]

with open(output_path, 'w') as f:
    json.dump(data, f, indent=2)

print(f'saved {len(data)} solutions to {output_path}')

### Check archive for true non dominance

In [1]:
import json


def check_non_dominated(path):
    """Verify that every solution in the JSON archive at `path` is non-dominated.

    All 4 objectives are treated as bigger-is-better
    (reduction, coverage, fairness, -n_facilities).
    Returns True if the archive is a valid Pareto front.
    """
    with open(path) as f:
        data = json.load(f)

    n = len(data)
    objs = [tuple(entry['obj']) for entry in data]

    def dominates(a, b):
        return all(x >= y for x, y in zip(a, b)) and any(x > y for x, y in zip(a, b))

    # Find any entry dominated by another (one dominator is enough to fail it)
    dominated = []
    for i in range(n):
        for j in range(n):
            if i != j and dominates(objs[j], objs[i]):
                dominated.append((i, j))
                break

    # Duplicate objective vectors: not a domination violation (no strict >),
    # but redundant in a proper archive
    seen = {}
    duplicates = []
    for i, obj in enumerate(objs):
        if obj in seen:
            duplicates.append((seen[obj], i))
        else:
            seen[obj] = i

    print(f'checked {n} solutions from {path}')
    print(f'  dominated:             {len(dominated)}')
    print(f'  duplicate obj vectors: {len(duplicates)}')

    if dominated:
        print(f'\ndominated entries:')
        for i, j in dominated[:20]:
            print(f'  [{i}] {objs[i]}  <-  dominated by [{j}] {objs[j]}')
        if len(dominated) > 20:
            print(f'  ...and {len(dominated) - 20} more')

    if duplicates:
        print(f'\nduplicate obj vectors (valid but redundant):')
        for i, j in duplicates[:10]:
            print(f'  [{i}] and [{j}]: {objs[i]}')

    return len(dominated) == 0


# Change for whatever file you want to check
check_non_dominated('my_archive.json')

checked 43 solutions from my_archive.json
  dominated:             0
  duplicate obj vectors: 0


True

### Hypervolume on JSON

In [ ]:
# Bash command to install pymoo
# pip install pymoo

In [3]:
import json
import numpy as np
from pymoo.indicators.hv import HV


# --- config ---
archive_path = 'my_archive.json'   # change to whichever archive you're scoring

# reference point: 0 on every axis, dominated by every real solution.
# for a fair hand comparison, whoever you're comparing against must use the same ref_point.
ref_point = np.array([0.0, 0.0, 0.0, 5.0])


# --- load ---
with open(archive_path) as f:
    data = json.load(f)

F = np.array([entry['obj'] for entry in data])
# columns: reduction, coverage, fairness, -n_facilities


# --- diagnostics for the hand comparison ---
print(f'file:           {archive_path}')
print(f'solutions:      {len(F)}')
print(f'reduction max:  {F[:, 0].max():.2f}')
print(f'coverage max:   {F[:, 1].max():.2f}')
print(f'fairness max:   {F[:, 2].max():.2f}')
print(f'k range:        {-int(F[:, 3].max())} to {-int(F[:, 3].min())} facilities')


# --- hypervolume (pymoo minimises, so negate) ---
hv = HV(ref_point=ref_point)(-F)
print(f'\nhypervolume:    {hv:.4e}')
print(f'ref point used: {ref_point.tolist()}')

file:           my_archive.json
solutions:      43
reduction max:  11845.49
coverage max:   24840.00
fairness max:   12470.10
k range:        2 to 4 facilities

hypervolume:    6.3803e+12
ref point used: [0.0, 0.0, 0.0, 5.0]
